# Dogs vs. Cats Image Classification — EfficientNetB4 Transfer Learning

**Competition:** Dogs vs. Cats Redux: Kernels Edition  

---

## Problem Statement

Given 25,000 labeled images of cats and dogs, the objective is to build a binary image classifier that predicts whether a new, unseen image contains a dog or a cat. Model predictions are evaluated using log loss on the Kaggle private leaderboard.

---

## Methodology

This solution applies Transfer Learning using EfficientNetB4, a convolutional neural network pretrained on ImageNet (1.2 million images, 1,000 classes). The pretrained backbone is adapted to the dogs vs. cats task through a two-phase training strategy: frozen backbone training followed by selective fine-tuning at a reduced learning rate.

The pretrained backbone is first frozen while a new classification head is trained, then partially unfrozen for joint fine-tuning at a lower learning rate.

---

## Approach Summary

| Component | Implementation |
|-----------|---------------|
| Data preprocessing | Images organized into class subfolders, resized to 224x224, normalized using EfficientNetB4 `preprocess_input` |
| Model | EfficientNetB4 backbone with a two-phase training strategy (frozen head training, then selective fine-tuning) |
| Alternatives explored | VGG16 with a frozen backbone, compared against EfficientNetB4 two-phase fine-tuning |
| Data augmentation | Horizontal flip, vertical flip, rotation, zoom, brightness adjustment (training data only) |
| Overfitting handling | 20% validation split, Dropout(0.5), EarlyStopping, ReduceLROnPlateau |

---

## Step 0 — Environment Setup

We disable XLA compilation before importing TensorFlow. XLA causes training instability on Kaggle GPU where accuracy gets stuck at 0.50. This must be set before any TensorFlow import.

In [ ]:
import os

# Disable XLA compilation before TensorFlow is imported.
# XLA can cause training instability on Kaggle GPU sessions.
os.environ['TF_XLA_FLAGS']         = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print('Environment configured.')

In [ ]:
import shutil
import zipfile
import warnings
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import tensorflow        as tf

from PIL                                         import Image
from sklearn.metrics                             import log_loss, accuracy_score
from tensorflow                                  import keras
from tensorflow.keras                            import layers, models
from tensorflow.keras.preprocessing.image        import ImageDataGenerator
from tensorflow.keras.applications               import EfficientNetB4
from tensorflow.keras.applications.efficientnet  import preprocess_input
from tensorflow.keras.callbacks                  import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

warnings.filterwarnings('ignore')

# Fix random seeds for reproducibility across runs.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Enable GPU memory growth to prevent TensorFlow from
# allocating all GPU memory at once.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Disable XLA JIT compilation at the TensorFlow level as well.
tf.config.optimizer.set_jit(False)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs available     : {tf.config.list_physical_devices("GPU")}')

---

## Step 1 — Data Preparation

### 1.1 Unzip Competition Files

The competition provides `train.zip` (25,000 labeled images) and `test.zip` (12,500 unlabeled images). We extract both into the working directory.

In [ ]:
BASE_PATH = '/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition'

if not os.path.exists('/kaggle/working/train'):
    print('Extracting train.zip ...')
    with zipfile.ZipFile(f'{BASE_PATH}/train.zip', 'r') as z:
        z.extractall('/kaggle/working/')

    print('Extracting test.zip  ...')
    with zipfile.ZipFile(f'{BASE_PATH}/test.zip', 'r') as z:
        z.extractall('/kaggle/working/')
else:
    print('Data already extracted.')

n_train = len(os.listdir('/kaggle/working/train'))
n_test  = len(os.listdir('/kaggle/working/test'))

print(f'Training images : {n_train:,}')
print(f'Test images     : {n_test:,}')

### 1.2 Organise Into Class Subfolders

`flow_from_directory` requires images to be organised into one subfolder per class. We copy the flat training images into `cats/` and `dogs/` subfolders.

**Focus: Data Preprocessing — resizing, normalization, data loading **

Preprocessing applied:
- Images resized to 224 x 224 pixels (EfficientNetB4 standard input)
- Pixel values normalised using EfficientNetB4 `preprocess_input` (channel-wise mean subtraction — not simple division by 255)
- 80% training / 20% validation split applied via `validation_split`

In [ ]:
# Folder structure required by flow_from_directory:
#   /data/train/cats/  <- all cat images
#   /data/train/dogs/  <- all dog images

DATA_DIR  = '/kaggle/working/data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
CAT_DIR   = os.path.join(TRAIN_DIR, 'cats')
DOG_DIR   = os.path.join(TRAIN_DIR, 'dogs')

os.makedirs(CAT_DIR, exist_ok=True)
os.makedirs(DOG_DIR, exist_ok=True)

if len(os.listdir(CAT_DIR)) == 0:
    print('Organising images into class subfolders ...')
    for filename in os.listdir('/kaggle/working/train'):
        src = os.path.join('/kaggle/working/train', filename)
        dst = CAT_DIR if filename.startswith('cat') else DOG_DIR
        shutil.copy(src, dst)
    print('Done.')
else:
    print('Subfolders already populated.')

print(f'Cat images : {len(os.listdir(CAT_DIR)):,}')
print(f'Dog images : {len(os.listdir(DOG_DIR)):,}')

### 1.3 Data Exploration

Before building any model, we explore the data to understand image characteristics and class balance.

In [ ]:
# Display sample training images from both classes.

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Sample Training Images — Raw (Before Preprocessing)', fontsize=13, fontweight='bold')

cat_files = os.listdir(CAT_DIR)[:6]
dog_files = os.listdir(DOG_DIR)[:6]

for col, fname in enumerate(cat_files):
    img = Image.open(os.path.join(CAT_DIR, fname))
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Cat  {img.size[0]}x{img.size[1]}', fontsize=8, color='steelblue')
    axes[0, col].axis('off')

for col, fname in enumerate(dog_files):
    img = Image.open(os.path.join(DOG_DIR, fname))
    axes[1, col].imshow(img)
    axes[1, col].set_title(f'Dog  {img.size[0]}x{img.size[1]}', fontsize=8, color='coral')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('01_sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print('Observation: images have variable sizes and orientations.')
print('All images will be standardised to 224 x 224 pixels during loading.')

In [ ]:
# Class distribution chart.

n_cats = len(os.listdir(CAT_DIR))
n_dogs = len(os.listdir(DOG_DIR))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Class Distribution', fontsize=12, fontweight='bold')

bars = ax1.bar(['Cats', 'Dogs'], [n_cats, n_dogs],
               color=['steelblue', 'coral'], edgecolor='black', linewidth=0.8)
ax1.set_ylabel('Number of Images')
ax1.set_title('Image Count per Class')
ax1.set_ylim(0, 14000)
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [n_cats, n_dogs]):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 150,
             f'{val:,}', ha='center', fontweight='bold')

ax2.pie([n_cats, n_dogs], labels=['Cats', 'Dogs'],
        colors=['steelblue', 'coral'],
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Class Balance')

plt.tight_layout()
plt.savefig('02_class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print('Classes are perfectly balanced at 50/50.')
print('No class weighting or resampling is required.')

---

## Step 2 — Data Generators and Augmentation

**Focus: Data Augmentation — rotation, flip, etc. **

Data augmentation creates artificial variations of training images during each epoch. The model sees a slightly different version of each image every time, which prevents memorisation and improves generalisation to unseen data.

Augmentation is applied **only to training images**. Validation and test images are passed through preprocessing only — they are never augmented.

| Technique | Setting | Rationale |
|-----------|---------|----------|
| Horizontal flip | True | A cat or dog looks the same from either direction |
| Vertical flip | True | Handles upside-down or unusual angles |
| Rotation | 20 degrees | Photos are often taken at slight angles |
| Zoom | 20 percent | Animals appear at different distances |
| Brightness | 0.8 to 1.2 | Indoor and outdoor lighting varies |

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
VAL_SPLIT  = 0.20

# Training generator applies augmentation and EfficientNetB4 preprocessing.
# EfficientNetB4 was trained with channel-wise mean subtraction.
# We must use preprocess_input, not simple rescale=1/255.
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = VAL_SPLIT,
    horizontal_flip        = True,
    vertical_flip          = True,
    rotation_range         = 20,
    zoom_range             = 0.20,
    brightness_range       = [0.8, 1.2]
)

# Validation generator applies preprocessing only — no augmentation.
val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = VAL_SPLIT
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'training',
    seed        = SEED
)

val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'validation',
    shuffle     = False
)

print(f'Training images   : {train_generator.samples:,}')
print(f'Validation images : {val_generator.samples:,}')
print(f'Class mapping     : {train_generator.class_indices}')
print('Expected mapping  : cats = 0, dogs = 1')

In [ ]:
# Visualise augmentation effect.
# We take one cat image and one dog image and apply augmentation
# multiple times to show the range of transformations the model sees.

aug_preview = ImageDataGenerator(
    horizontal_flip  = True,
    vertical_flip    = True,
    rotation_range   = 20,
    zoom_range       = 0.20,
    brightness_range = [0.8, 1.2],
    rescale          = 1.0 / 255
)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle(
    'Data Augmentation — Six Random Transformations of the Same Image',
    fontsize=13, fontweight='bold'
)

for row, (folder, label, color) in enumerate([
        (CAT_DIR, 'Cat', 'steelblue'),
        (DOG_DIR, 'Dog', 'coral')
]):
    fname  = os.listdir(folder)[0]
    image  = np.array(Image.open(os.path.join(folder, fname)).resize((IMG_SIZE, IMG_SIZE)))
    image  = image.reshape((1,) + image.shape)
    gen    = aug_preview.flow(image, batch_size=1)

    for col in range(6):
        aug_img = next(gen)[0]
        axes[row, col].imshow(aug_img)
        axes[row, col].set_title(f'{label} — variant {col + 1}',
                                  fontsize=8, color=color)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('03_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()

print('Augmentation is applied only to training images.')
print('Validation and test images receive preprocessing only.')

---

## Step 3 — Main Model: EfficientNetB4 Transfer Learning

**Focus: Model Description — clear description of architecture **

### Why EfficientNetB4

EfficientNetB4 was pretrained on ImageNet — 1.2 million images across 1,000 classes. It has already learned to detect edges, textures, shapes, and complex object features. We remove its original 1,000-class output layer and replace it with our own binary dog/cat output.

### Architecture

```
Input Image (224 x 224 x 3 RGB)
            |
EfficientNetB4 Backbone  — pretrained ImageNet weights
   Phase 1 : theta1 FROZEN  — backbone weights locked
   Phase 2 : theta1 DEFROST — top 30 layers fine-tuned
            |
GlobalAveragePooling2D    — converts (7,7,1792) to (1792,)
            |
Dense(256, ReLU)          — learns dog/cat specific patterns
            |
Dropout(0.5)              — randomly disables 50% of neurons during training
            |
Dense(1, Sigmoid)         — outputs P(dog) between 0 and 1
```

### Two-Phase Training Strategy

| Phase | Backbone State | Learning Rate | Epochs | Purpose |
|-------|---------------|--------------|--------|----------|
| Phase 1 | Frozen (theta1 locked) | 0.001 | up to 10 | Train new head only — fast convergence |
| Phase 2 | Top 30 defrosted | 0.00001 | up to 10 | Fine-tune backbone — improve accuracy |

In [ ]:
# Load EfficientNetB4 pretrained on ImageNet.
# include_top=False removes the original 1000-class output layer.
# input_shape must match our resized images: 224 x 224 x 3.

base_model = EfficientNetB4(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_SIZE, IMG_SIZE, 3)
)

# Freeze the backbone for Phase 1.
# This is theta1 in the professor's notation.
# Gradient descent will not update these weights during Phase 1.
base_model.trainable = False

# Build the classification head.
# This is theta2 — trained from random initialisation.
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='EfficientNetB4_Dogs_vs_Cats')

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

total_params    = model.count_params()
trainable_p1    = sum(tf.size(v).numpy() for v in model.trainable_variables)
non_trainable   = total_params - trainable_p1

model.summary()
print(f'Total parameters        : {total_params:,}')
print(f'Trainable (Phase 1)     : {trainable_p1:,}  — new head only (theta2)')
print(f'Non-trainable (backbone): {non_trainable:,}  — frozen ImageNet weights (theta1)')

---

## Step 4 — Phase 1: Train Head Only

**Focus: Validation Strategy and Overfitting Handling **

The backbone is frozen. Only the new Dense head (theta2) is trained. This is fast and stable because we are not risking corruption of the pretrained weights.

Overfitting prevention measures:
- 20% validation split — model never sees these images during training
- Dropout(0.5) — randomly disables neurons to prevent co-adaptation
- EarlyStopping — stops training when validation loss stops improving
- ReduceLROnPlateau — reduces learning rate when training stalls

In [ ]:
# Callbacks for Phase 1 training.

early_stop_1 = EarlyStopping(
    monitor              = 'val_loss',
    patience             = 4,
    restore_best_weights = True,
    verbose              = 1
)

reduce_lr_1 = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.3,
    patience = 2,
    min_lr   = 1e-7,
    verbose  = 1
)

checkpoint_1 = ModelCheckpoint(
    filepath       = 'best_phase1.keras',
    monitor        = 'val_loss',
    save_best_only = True,
    verbose        = 0
)

print('Phase 1 — Training classification head with backbone frozen')
print(f'Validation split      : {int(VAL_SPLIT * 100)}%')
print(f'Dropout               : 0.50')
print(f'EarlyStopping         : patience = 4')
print(f'ReduceLROnPlateau     : factor = 0.3, patience = 2')
print()

history_phase1 = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs          = 10,
    callbacks       = [early_stop_1, reduce_lr_1, checkpoint_1],
    verbose         = 1
)

p1_best_val_acc  = max(history_phase1.history['val_accuracy'])
p1_best_val_loss = min(history_phase1.history['val_loss'])

print(f'Phase 1 complete.')
print(f'Best validation accuracy : {p1_best_val_acc * 100:.2f}%')
print(f'Best validation loss     : {p1_best_val_loss:.4f}')

---

## Step 5 — Phase 2: Fine-Tuning

Once the classification head has converged, the top 30 layers of the EfficientNetB4 backbone are unfrozen and retrained at a significantly lower learning rate. This allows the backbone to adapt its high-level feature representations to the dogs vs. cats domain while preserving the lower-level features acquired during ImageNet pretraining.

The learning rate for Phase 2 is set to 1e-5, which is 100 times smaller than Phase 1. This conservative update step ensures that the pretrained weights are refined rather than overwritten.

In [ ]:
# Unfreeze the backbone for fine-tuning.
base_model.trainable = True

# Keep early layers frozen — they detect universal low-level features
# such as edges and colours that do not need to change.
# Only unfreeze the final 30 layers which detect high-level patterns.
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

unfrozen_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f'Unfrozen backbone layers : {unfrozen_count} of {len(base_model.layers)}')

# Recompile with a much smaller learning rate.
# Using 1e-5 instead of 1e-3 ensures we make tiny adjustments
# without destroying the pretrained ImageNet weights.
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

early_stop_2 = EarlyStopping(
    monitor              = 'val_loss',
    patience             = 4,
    restore_best_weights = True,
    verbose              = 1
)

reduce_lr_2 = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.3,
    patience = 2,
    min_lr   = 1e-8,
    verbose  = 1
)

checkpoint_2 = ModelCheckpoint(
    filepath       = 'best_phase2.keras',
    monitor        = 'val_loss',
    save_best_only = True,
    verbose        = 0
)

print('Phase 2 — Fine-tuning top 30 backbone layers')
print(f'Learning rate : 1e-5  (100x smaller than Phase 1)')
print(f'EarlyStopping : patience = 4')
print()

history_phase2 = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs          = 10,
    callbacks       = [early_stop_2, reduce_lr_2, checkpoint_2],
    verbose         = 1
)

p2_best_val_acc  = max(history_phase2.history['val_accuracy'])
p2_best_val_loss = min(history_phase2.history['val_loss'])

print(f'Phase 2 complete.')
print(f'Best validation accuracy : {p2_best_val_acc * 100:.2f}%')
print(f'Best validation loss     : {p2_best_val_loss:.4f}')
print(f'Improvement from Phase 1 : +{(p2_best_val_acc - p1_best_val_acc) * 100:.2f}%')

---

## Step 6 — Training History Visualisation

We plot the combined training curves from Phase 1 and Phase 2 to show how the model improved over time and whether overfitting occurred.

In [ ]:
# Combine Phase 1 and Phase 2 history for a continuous view.

all_loss     = history_phase1.history['loss']         + history_phase2.history['loss']
all_val_loss = history_phase1.history['val_loss']     + history_phase2.history['val_loss']
all_acc      = history_phase1.history['accuracy']     + history_phase2.history['accuracy']
all_val_acc  = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
phase1_end   = len(history_phase1.history['loss'])
epochs_range = range(1, len(all_loss) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EfficientNetB4 Training History — Phase 1 (Frozen) and Phase 2 (Fine-tuned)',
             fontsize=12, fontweight='bold')

ax1.plot(epochs_range, all_loss,     color='steelblue', linewidth=2,
         linestyle='-',  label='Training Loss')
ax1.plot(epochs_range, all_val_loss, color='coral',     linewidth=2,
         linestyle='--', label='Validation Loss')
ax1.axvline(x=phase1_end, color='gray', linestyle=':', linewidth=1.5,
            label='Phase 1 to Phase 2')
ax1.set_title('Loss (Binary Cross-Entropy)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, all_acc,     color='steelblue', linewidth=2,
         linestyle='-',  label='Training Accuracy')
ax2.plot(epochs_range, all_val_acc, color='coral',     linewidth=2,
         linestyle='--', label='Validation Accuracy')
ax2.axvline(x=phase1_end, color='gray', linestyle=':', linewidth=1.5,
            label='Phase 1 to Phase 2')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('04_training_history.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Step 7 — Generate Kaggle Submission

We load the best weights saved during Phase 2 and generate predictions for all 12,500 test images. The submission file contains two columns: `id` (image number) and `label` (predicted probability of being a dog).

In [ ]:
# Load the best model weights from Phase 2.
model.load_weights('best_phase2.keras')
print('Best Phase 2 weights loaded.')

# Test generator — same preprocessing as training, no augmentation, no shuffle.
# shuffle=False is critical: prediction order must match file ID order.
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = test_datagen.flow_from_directory(
    '/kaggle/working/',
    classes     = ['test'],
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = 1,
    shuffle     = False,
    class_mode  = None
)

print('Generating predictions on test images ...')
predictions = model.predict(test_generator, verbose=1)

# Extract numeric IDs from filenames.
# Filenames look like: test/1234.jpg — we extract the number 1234.
filenames = test_generator.filenames
ids       = [int(fname.split('/')[1].split('.')[0]) for fname in filenames]

submission = pd.DataFrame({
    'id'   : ids,
    'label': predictions.flatten()
})

# Sort by ID as required by the Kaggle submission format.
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('submission.csv saved.')
print(f'Total rows     : {len(submission):,}')
print(f'Min prediction : {predictions.min():.6f}')
print(f'Max prediction : {predictions.max():.6f}')
print(f'Mean prediction: {predictions.mean():.4f}')
print()
print('First 10 rows:')
print(submission.head(10).to_string(index=False))

---

## Step 8 — Validation Score Estimate

Before submitting, we estimate the Kaggle log loss score using the validation set, for which we know the true labels.

In [ ]:
# Generate predictions on the validation set to estimate Kaggle score.
val_predictions = model.predict(val_generator, verbose=1).flatten()
val_true_labels = val_generator.classes

val_log_loss = log_loss(val_true_labels, val_predictions)
val_accuracy = accuracy_score(val_true_labels,
                               (val_predictions > 0.5).astype(int))

print('Validation Performance Estimate')
print('================================')
print(f'Log Loss : {val_log_loss:.5f}')
print(f'Accuracy : {val_accuracy * 100:.2f}%')
print()
print('Expected Kaggle points based on log loss:')
score_table = [
    (0.050, 40), (0.055, 39), (0.060, 38), (0.065, 37),
    (0.070, 36), (0.075, 35), (0.080, 34), (0.100, 33),
    (0.150, 32), (0.200, 31), (0.500, 30)
]
for threshold, points in score_table:
    marker = '  <-- estimated position' if val_log_loss < threshold else ''
    print(f'  log loss < {threshold:.3f}  :  {points} points{marker}')

---

## Step 9 — Alternative Model: VGG16 Transfer Learning

**Focus: Exploration of Alternatives — tried different model settings or architectures **

To demonstrate exploration of alternative architectures, we build a second transfer learning model using VGG16, a simpler 16-layer network also pretrained on ImageNet. VGG16 is trained with its backbone fully frozen — only the classification head is updated. This serves as a direct comparison against the two-phase EfficientNetB4 approach.

**Architecture:**
```
Input (224 x 224 x 3)
VGG16 Backbone — fully frozen ImageNet weights
GlobalAveragePooling2D
Dense(256, ReLU)
Dropout(0.5)
Dense(1, Sigmoid)
```

In [ ]:
# Alternative model: VGG16 with frozen backbone.
# VGG16 is a 16-layer network pretrained on ImageNet.
# We freeze all backbone layers and train only the classification head.
# This is faster than training from scratch and provides a meaningful
# comparison against EfficientNetB4.

from tensorflow.keras.applications      import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

# VGG16 requires its own preprocessing function.
vgg_train_datagen = ImageDataGenerator(
    preprocessing_function = vgg_preprocess,
    validation_split       = VAL_SPLIT
)

vgg_train_gen = vgg_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'training',
    seed        = SEED
)

vgg_val_gen = vgg_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'validation',
    shuffle     = False
)

# Load VGG16 pretrained on ImageNet with backbone fully frozen.
vgg_base = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_SIZE, IMG_SIZE, 3)
)
vgg_base.trainable = False

vgg_model = models.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='VGG16_Frozen_Baseline')

vgg_model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

vgg_model.summary()

print('Training VGG16 with frozen backbone — 5 epochs ...')
history_vgg = vgg_model.fit(
    vgg_train_gen,
    validation_data = vgg_val_gen,
    epochs          = 5,
    callbacks       = [EarlyStopping(monitor='val_loss', patience=3,
                                     restore_best_weights=True, verbose=1)],
    verbose         = 1
)

vgg_best_val_acc  = max(history_vgg.history['val_accuracy'])
vgg_best_val_loss = min(history_vgg.history['val_loss'])

print(f'VGG16 (frozen) — Best Validation Accuracy : {vgg_best_val_acc * 100:.2f}%')
print(f'VGG16 (frozen) — Best Validation Loss     : {vgg_best_val_loss:.4f}')

---

## Step 10 — Model Comparison

We compare VGG16 with frozen backbone against EfficientNetB4 with two-phase fine-tuning to demonstrate that selective unfreezing and fine-tuning yields superior results.

In [ ]:
# Side-by-side comparison of all three model configurations.

model_names = ['VGG16\n(Frozen)', 'EfficientNetB4\nPhase 1', 'EfficientNetB4\nPhase 2']
accuracies  = [vgg_best_val_acc, p1_best_val_acc, p2_best_val_acc]
bar_colors  = ['#d9534f', '#5bc0de', '#5cb85c']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, [a * 100 for a in accuracies],
              color=bar_colors, edgecolor='black', linewidth=0.8, width=0.5)

ax.set_title('Model Comparison — Validation Accuracy',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Validation Accuracy (%)')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f'{acc * 100:.1f}%',
        ha='center', fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('05_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Model Comparison Summary')
print('========================')
print(f'VGG16 frozen backbone         : {vgg_best_val_acc * 100:.2f}%')
print(f'EfficientNetB4 — Phase 1      : {p1_best_val_acc * 100:.2f}%')
print(f'EfficientNetB4 — Phase 2      : {p2_best_val_acc * 100:.2f}%  (final model)')
print()
print('Conclusion: EfficientNetB4 with fine-tuning outperforms VGG16 frozen baseline.')
print('Two-phase transfer learning yields the best performance on this task.')

---

## Step 11 — Submitting to Kaggle

1. Download `submission.csv` from `/kaggle/working`
2. Go to the competition page and click Submit Predictions
3. Upload `submission.csv`
4. Wait ~2 minutes for scoring, then check the Private Score under My Submissions

---

## Solution Summary

| Component | Choice | Justification |
|-----------|--------|---------------|
| Main model | EfficientNetB4 | Pretrained on 1.2 million ImageNet images |
| Alternative model | VGG16 frozen backbone | Faster alternative — compares different pretrained architectures |
| Training strategy | Two-phase freeze and defrost | Standard transfer learning approach: adapt a pretrained backbone without disturbing its learned features early on |
| Preprocessing | EfficientNetB4 preprocess_input | Required channel-wise normalisation for this architecture |
| Data loading | flow_from_directory with class subfolders | Clean and reliable — no label parsing from filenames |
| Augmentation | Flip, rotation, zoom, brightness | Improves generalisation on limited training data |
| Validation split | 20% held out | Monitors overfitting throughout training |
| Regularisation | Dropout(0.5) | Primary regularisation in classification head |
| Alternative placement | VGG16 runs after main model | Prevents GPU memory contamination between models |